# 1. Import thư viện

In [1]:
"""
Các dòng code bên dưới dùng để import các thư viện thiết yếu để xây dựng mô hình RAG
"""
import json
import sys
import uuid
import os
import time
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_core.runnables import RunnableLambda
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.documents import Document
from underthesea import word_tokenize
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler

# 2. Chunking các metadata của sản phẩm thời trang

In [2]:
def process_fashion_metadata(file_path):
    """
    Hàm này được sử dụng để đọc file JSONL và chuyển đổi thành danh sách các LangChain Documents
    """
    documents = []

    # Mở và đọc từng dòng trong file JSONL
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): # Kiểm tra các dòng trống và bỏ qua nó
                continue
            item = json.loads(line) # Load từng dòng json

            # Bước 1: Xử lý metadata sản phẩm để lấy ra các trường quan trọng và đặt nó cùng với "Bước 2"
            # Trích xuất danh sách link ảnh từ mảng images
            image_urls = [img.get("large") for img in item.get("images", []) if "large" in img]

            # Lấy các thông tin metadata của sản phẩm 
            metadata = {
                "product_id": item.get("product_id", ""),
                "category": item.get("category", ""),
                "department": item.get("department", ""),
                "brand": item.get("brand", ""),
                "price": item.get("price", 0),
                "images": image_urls
            }

            # Bước 2: Xử lý theo hướng văn bản hóa dữ liệu có cấu trúc
            # Lấy thông tin details từ metadata của sản phẩm
            details = item.get("details", {})
            main_color = details.get("main_color", "Không có thông tin về màu sắc")
            material = details.get("material", "Không có thông tin về chất liệu")
            size = details.get("size", "Không có thông tin về kích thước")
            pattern = details.get("pattern", "Không có thông tin về họa tiết")
            
            # Khung văn bản mô tả sản phẩm
            page_content = (
                f"Sản phẩm {item.get('title', 'Không có tên sản phẩm')} thuộc danh mục {item.get('category', '')} "
                f"dành cho {item.get('department', '')}. "
                f"Thương hiệu: {item.get('brand', 'Không có thông tin về thương hiệu')}. "
                f"Mức giá: {item.get('price', 0)} VNĐ. "
                f"Đặc điểm chi tiết: có các loại màu sắc: {main_color}, chất liệu: {material}, kích cỡ: {size}, họa tiết: {pattern}. "
                f"Phù hợp sử dụng cho {item.get('season', '')} và các dịp {item.get('occasion', '')}. "
                f"Mô tả chi tiết: {item.get('description', '')}"
            )
            
            # Bước 3: Tạo LangChain Documents
            doc = Document(page_content=page_content, metadata=metadata)
            documents.append(doc)
            """
            Sau khi tạo xong LangChain Documents thì đối tượng Document nó sẽ có 2 thành phần sau:
            - page_content (string): Đây chính là phần văn bản đã được "văn bản hóa" từ dữ liệu thô có cấu trúc ở "Bước 2". Phần này sau đó sẽ
              đi qua một mô hình Embedding để tạo vector ngữ nghĩa.
            - metadata (dict): Đây là một từ điển chứa các thông tin bổ trợ. Phần này sẽ được giữ nguyên cấu trúc của nó.
            """

        return documents

def process_all_directory(directory_path):
    """
    Hàm lặp qua tất cả các file .jsonl trong một thư mục và gộp data lại.
    """
    all_documents = []
    
    # Duyệt qua tất cả các file có trong thư mục
    for filename in sorted(os.listdir(directory_path)):
        # Chỉ xử lý những file có đuôi là .jsonl
        if filename.endswith(".jsonl"):
            # Tạo đường dẫn tuyệt đối cho từng file
            file_path = os.path.join(directory_path, filename)
            
            print(f"[THÔNG BÁO] Đang đọc file: {filename}...")
            
            # Gọi lại hàm process_fashion_metadata(file_path) đã được định nghĩa ở phía trên
            docs = process_fashion_metadata(file_path)
            
            # LƯU Ý QUAN TRỌNG: Dùng .extend() để gộp các list lại với nhau
            # Không dùng .append() vì nó sẽ tạo ra list lồng list
            all_documents.extend(docs)
            
    return all_documents

# 3. Sử dụng mô hình Embedding để xử lý các Chunks vào lưu vào Qdrant vector DB

In [3]:
class VietnameseSegmentedEmbeddings(Embeddings):
    # Hàm này dùng để khởi tạo mô hình Embedding
    def __init__(self, model_name="bkai-foundation-models/vietnamese-bi-encoder"):
        self.hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)
        
    # Hàm này nhận đầu vào là text và dùng word_tokenize của underthesea để thực hiện tách từ và trả về từ đã tách
    def _segment_text(self, text: str) -> str:
        if not text:
            return ""
        return word_tokenize(text, format="text")

    # Hàm này sử dụng mô hình embedding đã khởi tạo để vector hóa dữ liệu
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        segmented_texts = [self._segment_text(text) for text in texts]
        return self.hf_embeddings.embed_documents(segmented_texts)

    # Hàm này sử dụng mô hình embedding đã khởi tọa để vector hóa câu truy vấn của người dùng
    def embed_query(self, text: str) -> list[float]:
        segmented_text = self._segment_text(text)
        return self.hf_embeddings.embed_query(segmented_text)

In [4]:
"""
Đoạn code bên dưới sẽ hoạt động theo các bước như sau:
- Bước 1: Trỏ đến thư mục chứa các metadata sản phẩm thời trang dưới dạng .jsonl, sau đó sẽ quét toàn bộ file và thực hiện chunking từng file một.
- Bước 2: Sau khi setup thành công Qdrant trên Docker, thì kết nối tới Qdrant đó thông qua địa chỉ ip local được cung cấp. Sau đó, tạo một collections 
          mới và thiết lập thông số cho vectors được lưu.
- Bước 3: Tiếp tục khởi tạo một VectorStore. VectorStore có nhiệm vụ số hóa các chunk tài liệu và lưu nó vào trong vector database đã được thiết lập.
"""
if __name__ == "__main__":
    # Trỏ đến thư mục chứa tất cả các file JSONL của bạn
    folder_path = "./Fashion_Metadata"

    # Khởi tạo Custom Embedding
    custom_embeddings = VietnameseSegmentedEmbeddings()

    if os.path.exists(folder_path):
        print(f"[THÔNG BÁO] Bắt đầu quét thư mục: {folder_path}\n" + "="*50)
        
        # Gọi hàm xử lý toàn bộ
        all_docs = process_all_directory(folder_path)
        
        print("="*50)
        print(f"[THÔNG BÁO] HOÀN TẤT! Tổng số lượng chunk (documents) thu được: {len(all_docs)}")

        qdrant_url = "http://localhost:6333"
        collection_name = "fashion_products"
        
        print(f"[THÔNG BÁO] Bắt đầu đẩy dữ liệu vào Qdrant tại: {qdrant_url}")
        
        client = QdrantClient(url=qdrant_url)
        
        # 1. KIỂM TRA SỐ LƯỢNG ĐÃ LƯU TRONG DATABASE
        if not client.collection_exists(collection_name):
            print(f"[THÔNG BÁO] Tạo mới collection '{collection_name}'...")
            client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(size=768, distance=Distance.COSINE),
            )
            current_count = 0 # Chưa có gì
        else:
            # Lấy thông tin database hiện tại
            collection_info = client.get_collection(collection_name)
            current_count = collection_info.points_count # Số lượng vector đang có sẵn
            print(f"[THÔNG BÁO] Tìm thấy {current_count} sản phẩm đã được lưu trước đó.")
        
        # 2. CẮT BỎ PHẦN DỮ LIỆU ĐÃ LÀM XONG
        # Lấy từ vị trí current_count cho đến hết
        remaining_docs = all_docs[current_count:]
        
        if len(remaining_docs) == 0:
            print("\n[THÔNG BÁO] 🎉 Toàn bộ dữ liệu đã được xử lý xong từ trước. Không cần chạy lại!")
        else:
            print(f"[THÔNG BÁO] Bắt đầu nhúng và lưu {len(remaining_docs)} sản phẩm còn lại...")
            
            # Khởi tạo VectorStore
            vector_db = QdrantVectorStore(
                client=client,
                collection_name=collection_name,
                embedding=custom_embeddings,
            )
            
            batch_size = 128
            
            # 3. CHẠY THANH TIẾN TRÌNH CHO PHẦN CÒN LẠI
            # Tham số `initial` giúp thanh tiến trình không bắt đầu từ 0, mà hiển thị đúng phần trăm thực tế
            with tqdm(total=len(all_docs), initial=current_count, desc="Tiến độ Vector hóa", unit="SP") as pbar:
                for i in range(0, len(remaining_docs), batch_size):
                    # Cắt lô dữ liệu tiếp theo
                    batch = remaining_docs[i : i + batch_size]
                    
                    # Đẩy vào Qdrant
                    vector_db.add_documents(documents=batch)
                    
                    # Cập nhật thanh tiến trình
                    pbar.update(len(batch))
        
            print("\n[THÔNG BÁO] 🎉 ĐÃ LƯU HOÀN TẤT VÀO VECTOR DATABASE!")
        
    else:
        print(f"[THÔNG BÁO] Không tìm thấy thư mục {folder_path}. Vui lòng kiểm tra lại.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--bkai-foundation-models--vietnamese-bi-encoder. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

[THÔNG BÁO] Bắt đầu quét thư mục: ./Fashion_Metadata
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Ao.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Ao_khoac.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Bong_tai.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Chan_vay.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Dam.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Day_chuyen.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Do_lot.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Dong_ho.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Gang_tay.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Ghim_cai_ao.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Giay.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Jumpsuit.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Khac.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Kinh_mat.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metadata_Mu.jsonl...
[THÔNG BÁO] Đang đọc file: Fashion_Metad

Tiến độ Vector hóa: 100%|██████████| 48938/48938 [2:27:57<00:00,  5.51SP/s]  


[THÔNG BÁO] 🎉 ĐÃ LƯU HOÀN TẤT VÀO VECTOR DATABASE!


# 4. Xây dựng Retriever để truy hồi dữ liệu

In [5]:
def load_vector_db():
    print("[THÔNG BÁO] Đang khởi tạo mô hình Embedding...")
    custom_embeddings = VietnameseSegmentedEmbeddings()
    
    print("[THÔNG BÁO] Đang kết nối tới Qdrant Database...")
    qdrant_url = "http://localhost:6333"
    client = QdrantClient(url=qdrant_url)
    
    # Khởi tạo QdrantVectorStore và gán vào biến vector_db
    vector_db = QdrantVectorStore(
        client=client,
        collection_name="fashion_products",
        embedding=custom_embeddings
    )
    
    print("✅ Đã kết nối thành công!")
    return vector_db

# Sử dụng trong file Chatbot
if __name__ == "__main__":
    vector_db = load_vector_db()

    # Thiết lập retriever để truy hồi thông tin
    retriever = vector_db.as_retriever(
        search_type="similarity_score_threshold", # Tìm kiếm theo độ tương đồng Vector dựa trên một ngưỡng
        search_kwargs={"k": 5, # "k" là số lượng kết quả bạn muốn lấy ra
                       "score_threshold": 0.7 # Chỉ lấy ra các sản phẩm có độ tương đồng Vector từ 70% trở lên
                      }    
    )
    
    print("[THÔNG BÁO] Đã khởi tạo Retriever thành công!")

[THÔNG BÁO] Đang khởi tạo mô hình Embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[THÔNG BÁO] Đang kết nối tới Qdrant Database...
✅ Đã kết nối thành công!
[THÔNG BÁO] Đã khởi tạo Retriever thành công!


# 5. Khởi tạo LLM để tích hợp vào hệ thống Chatbot

In [7]:
print("[THÔNG BÁO] Đang khởi tạo mô hình Qwen local...")
llm = ChatOllama(
    model="qwen2.5:3b-instruct",
    temperature=0.4,
    timeout=120,
    num_predict=350,
    num_ctx=8192,
)
llm_cau_hoi = ChatOllama(
    model="qwen2.5:3b-instruct",
    temperature=0.1,
    timeout=120,
    num_predict=128,
    num_ctx=8192,
)
print("[THÔNG BÁO] Đã khởi tạo LLM thành công!")

[THÔNG BÁO] Đang khởi tạo mô hình Qwen local...
[THÔNG BÁO] Đã khởi tạo LLM thành công!


# 6. Cấu hình Prompt và Redis

In [8]:
# 1. Kịch bản chính cho Chatbot
system_prompt = """Bạn là chuyên viên tư vấn thời trang cao cấp của shop.

QUY TẮC TỐI CAO (CHỐNG BỊA ĐẶT - ANTI-HALLUCINATION):
1. CHỈ SỬ DỤNG thông tin có trong phần "DỮ LIỆU SẢN PHẨM" bên dưới để trả lời.
2. TUYỆT ĐỐI KHÔNG tự bịa ra tên sản phẩm, không bịa ID, giá tiền, màu sắc hay chất liệu. 
3. NẾU KHÔNG TÌM THẤY SẢN PHẨM KHỚP NHU CẦU: Khẳng định ngay là shop chưa có mẫu này, xin lỗi và mời khách xem mẫu khác. Không được cố gắng đoán hoặc bịa ra sản phẩm.

QUY TẮC TRÍCH XUẤT VÀ ĐỊNH DẠNG:
4. BẮT BUỘC TRÍCH XUẤT CHÍNH XÁC 100%: Khi giới thiệu một sản phẩm, bạn PHẢI copy y nguyên trường [Sản phẩm] và [MÃ_SP] từ dữ liệu. Không thêm, không bớt, không dịch thuật.
5. TRÌNH BÀY BẮT BUỘC theo khuôn mẫu sau:
   - **[TÊN_SP]** (Mã SP: [MÃ_SP])
   - Giá: [GIÁ_TIỀN] VNĐ
   - Đặc điểm chi tiết: Lấy ra các thông tin về đặc điểm chi tiết của sản phẩm (màu sắc, chất liệu, kích cỡ, họa tiết) và dựa vào nó để VIẾT THÀNH MỘT CÂU HOÀN CHỈNH, KHÔNG ĐƯỢC BỊA ĐẶT THÔNG TIN.
   - (Viết 1-2 câu tư vấn thân thiện, khen ngợi sản phẩm dựa trên đặc điểm và mô tả chi tiết của nó)
6. GIỚI HẠN: Trả lời tự nhiên, thân thiện nhưng không vượt quá 250 từ.

DỮ LIỆU SẢN PHẨM:
{context}"""

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"), 
    ("human", "{input}")
])

REDIS_URL = "redis://localhost:6379"
SESSION_ID = str(uuid.uuid4()) 

def get_message_history(session_id: str):
    history = RedisChatMessageHistory(session_id, url=REDIS_URL)
    current_messages = history.messages
    if len(current_messages) > 6: 
        kept_messages = current_messages[-6:]
        history.clear()
        history.add_messages(kept_messages)
    return history

print("[THÔNG BÁO] Đã cấu hình Kịch bản chuẩn ChatML và Redis thành công!")

[THÔNG BÁO] Đã cấu hình Kịch bản chuẩn ChatML và Redis thành công!


# 7. Xây dựng kiến trúc RAG

In [9]:
print("[THÔNG BÁO] Đang lắp ráp Pipeline Retriever Thông minh...")

# 1. KỊCH BẢN PHỤ: Dạy AI cách viết lại câu hỏi
contextualize_q_system_prompt = """
BẠN LÀ QUERY REWRITER (CÔNG CỤ VIẾT LẠI TRUY VẤN). KHÔNG PHẢI CHATBOT.

Nhiệm vụ:
Đọc "Lịch sử trò chuyện" và "Câu hỏi mới". Viết lại câu hỏi mới thành một câu truy vấn tìm kiếm ĐỘC LẬP, RÕ RÀNG.

=====================
QUY TẮC BẮT BUỘC:

1. THAY THẾ ĐẠI TỪ: Nếu câu hỏi mới có chứa các từ ám chỉ (ví dụ: "cái áo đó", "sản phẩm này", "món đầu tiên", "nó"), bạn BẮT BUỘC phải tìm TÊN SẢN PHẨM CỤ THỂ trong Lịch sử trò chuyện để thay thế vào.
2. CHỈ trả về JSON với định dạng duy nhất:
{{"query": "<câu truy vấn đã viết lại>"}}
3. KHÔNG trả lời, KHÔNG giải thích, KHÔNG thêm thông tin thừa.
=====================
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_prompt_template = """
[MÃ_SP: {product_id}]
THÔNG TIN CHI TIẾT: {page_content}
"""
doc_prompt = PromptTemplate.from_template(document_prompt_template)

import re # Thêm thư viện regex nếu cần

def safe_parse(output, original_query):
    # 1. In ra màn hình để bạn dễ dàng bắt bệnh (Debug)
    print(f"\n[DEBUG - LLM RAW OUTPUT]: {output}\n")
    
    # 2. Xóa bỏ thẻ markdown (```json và ```) để hàm json.loads không bị lỗi
    cleaned_output = output.strip()
    if cleaned_output.startswith("```json"):
        cleaned_output = cleaned_output[7:]
    elif cleaned_output.startswith("```"):
        cleaned_output = cleaned_output[3:]
    if cleaned_output.endswith("```"):
        cleaned_output = cleaned_output[:-3]
    
    cleaned_output = cleaned_output.strip()

    try:
        data = json.loads(cleaned_output)
        query = data.get("query", "").strip()

        if not query:
            return original_query

        # chặn hallucination kiểu trả lời
        blacklist = ["dưới đây", "gợi ý", "1.", "-", "**", ":"]
        if any(b in query.lower() for b in blacklist):
            return original_query

        # chặn query quá dài (thường là answer)
        if len(query.split()) > 30:
            return original_query

        return query
    except Exception as e:
        # In ra lỗi nếu parse JSON vẫn thất bại
        print(f"[DEBUG - JSON ERROR]: {e}") 
        return original_query

def rewrite_query(inputs):
    prompt_value = contextualize_q_prompt.invoke(inputs)
    raw_output = llm_cau_hoi.invoke(prompt_value)

    rewritten_query = safe_parse(raw_output.content, inputs["input"])

    return rewritten_query

# 2. KHỞI TẠO CÁC CHUỖI RAG
# Chuỗi 1: Trợ lý viết lại câu hỏi
# chain rewrite → retriever
history_aware_retriever = RunnableLambda(rewrite_query) | retriever

# Chuỗi 2: Nhồi dữ liệu từ Qdrant vào Kịch bản chính (QA_PROMPT ở Cell 6)
document_chain = create_stuff_documents_chain(llm=llm, prompt=QA_PROMPT, document_prompt=doc_prompt)

# Chuỗi 3: Ghép nối Trợ lý viết lại câu vào hệ thống
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

# 4. TRÙM REDIS ĐỂ LƯU LỊCH SỬ
full_chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

print("[THÔNG BÁO] Hệ thống Retriever Thông minh đã sẵn sàng!")

[THÔNG BÁO] Đang lắp ráp Pipeline Retriever Thông minh...
[THÔNG BÁO] Hệ thống Retriever Thông minh đã sẵn sàng!


# 8. Luồng chạy chính

In [10]:
SESSION_ID = str(uuid.uuid4()) 
# 1. TẠO "CAMERA" GIÁM SÁT RETRIEVER
class SpyRetrieverHandler(BaseCallbackHandler):
    # Hàm này sẽ tự động kích hoạt ngay trước khi Qdrant đi tìm dữ liệu
    def on_retriever_start(self, serialized: dict, query: str, **kwargs):
        # 'query' ở đây chính là thành phẩm đã được llm viết lại dựa trên lịch sử
        print(f"\n🕵️‍♂️ [AI ĐÃ VIẾT LẠI CÂU HỎI THÀNH]: {query}\n")

print("="*60)
print(" 👗👔 CHATBOT TƯ VẤN THỜI TRANG ĐÃ SẴN SÀNG 👔👗 ")
print("       (Nhập '0' rồi Enter để kết thúc)       ")
print("="*60 + "\n")

# Vòng lặp vô tận (cho đến khi gặp lệnh break)
while True:
    user_input = input("👤 Bạn: ")
    
    if user_input.strip() == "0":
        print("\n🤖 Chatbot: Dạ, cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại bạn nhé!")
        break
        
    if not user_input.strip():
        continue
        
    print("🤖 Chatbot: ", end="")

    start_time = time.time()
    first_token_time = None
    
    try:
        # 2. GẮN CAMERA VÀO LUỒNG STREAM
        for chunk in full_chat_chain.stream(
            {"input": user_input},
            config={
                "configurable": {"session_id": SESSION_ID},
                "callbacks": [SpyRetrieverHandler()]  # <--- Gắn camera vào đây
            }
        ):
            if "answer" in chunk:
                if first_token_time is None:
                    first_token_time = time.time()
                print(chunk["answer"], end="", flush=True)

        end_time = time.time()
        
        # Đề phòng trường hợp lỗi không sinh ra chữ nào
        if first_token_time is None:
            first_token_time = end_time
            
        ttft = first_token_time - start_time
        total_time = end_time - start_time
        
        # In bảng thống kê nhỏ sau mỗi câu trả lời
        print(f"\n\n⏱️ [THỐNG KÊ TỐC ĐỘ]:")
        print(f"   - Nhả chữ đầu tiên sau: {ttft:.2f} giây")
        print(f"   - Hoàn thành toàn bộ sau: {total_time:.2f} giây")
                
    except Exception as e:
        print(f"\n[LỖI HỆ THỐNG] Đã xảy ra sự cố: {e}")
        
    print("\n\n" + "-"*60 + "\n")

 👗👔 CHATBOT TƯ VẤN THỜI TRANG ĐÃ SẴN SÀNG 👔👗 
       (Nhập '0' rồi Enter để kết thúc)       

🤖 Chatbot: 
[DEBUG - LLM RAW OUTPUT]: {"query": "xin chào bạn"}


🕵️‍♂️ [AI ĐÃ VIẾT LẠI CÂU HỎI THÀNH]: xin chào bạn



No relevant docs were retrieved using the relevance score threshold 0.7


Xin chào! Tôi có thể giúp gì cho bạn hôm nay?

⏱️ [THỐNG KÊ TỐC ĐỘ]:
   - Nhả chữ đầu tiên sau: 50.55 giây
   - Hoàn thành toàn bộ sau: 52.36 giây


------------------------------------------------------------


🤖 Chatbot: Dạ, cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại bạn nhé!
